# Intraday test: how much of the morning move do we keep? (Juliet, Technical agent)

**Why:** in the competition, an order sent on trading day T (before ~14:00) is executed at the **T-1 close**.
So when we decide, we already know how the market moved **since that price**: the US afternoon after the Xetra
close (17:30 → 22:00), Asia overnight, and the European morning. If an ETF is already up at 13:00 and we buy at
yesterday's close, we capture that move, **if it does not reverse by the close**.

**Question:** does the move from the T-1 close to 13:00 on day T predict the return from the T-1 close to the
T close (and to the next valuation days)? How much of it is kept? Is buying the morning winners a real edge?

**How to run (Colab):** Runtime → Run all. No API key, no upload needed. It downloads ~2 years of free hourly
data from Yahoo (the maximum Yahoo gives for hourly bars), takes 2–5 minutes, and at the end downloads
`intraday_morning_test.csv` to your computer. Send that file (or the printed tables) back for the analysis.

In [ ]:
%pip install -q yfinance

In [ ]:
import time, numpy as np, pandas as pd, yfinance as yf
pd.set_option("display.width", 200)

DECISION_HOUR = 13      # we look at the market at 13:00 (deadline ~14:00): price = close of the 12:00-13:00 bar
TZ = "Europe/Paris"     # = Brussels / Berlin time

# Same Yahoo symbols as data/tickers.csv (Morgane, 8 fixed by hand)
TICKERS = {
 "WORLD": "EUNL.DE", "S&P 500": "SXR8.DE", "EUROPE 600": "LYP6.DE", "EURO STOXX 50": "SXRT.DE", "IBEX 35": "AMES.DE",
 "BEL 20": "BEL.BR", "WORLD SMALL CAP": "IUSN.DE", "WORLD MIN VOL": "IQQ0.DE", "DIVIDEND LEADERS": "VDIV.DE",
 "NASDAQ INCOME": "JEQP.DE", "NASDAQ": "SXRV.DE", "SEMIS": "SEC0.DE", "CYBERSEC": "USPY.DE", "BIOTECH": "NBTK.DE",
 "ROBOTICS": "2B76.DE", "SPACE": "JEDI.DE", "QUANTUM": "QUTM.DE", "EURO BANKS": "LYBK.DE", "HEALTH CARE": "XDWH.DE",
 "EMMKT": "IS3N.DE", "CHINA LC": "IQQC.DE", "INDIA": "QDV5.DE", "KOREA": "LKOR.DE", "BRAZIL": "IQQB.DE",
 "CHINA A": "36BZ.DE", "WORLD ENERGY": "XDW0.DE", "OIL & GAS E&P": "IS0D.DE", "URANIUM": "NUKL.DE",
 "CLEAN ENERGY": "RENW.DE", "HYDROGEN": "AMEE.DE", "COPPER MINERS": "4COP.DE", "RARE EARTHS": "VVMX.DE",
 "GOLD": "PPFB.DE", "GOLD MINERS": "G2X.DE", "SILVER MINERS": "SLVR.DE", "HIGHY BOND": "XHYG.DE",
 "US TREASURY 20Y": "DJAD.DE", "GLOBAL AGG BOND": "VAGF.DE", "EUR GOVT BOND": "EUNH.DE", "OVERNIGHT CASH": "LYOR.DE",
 "DEFENSE": "DFEN.DE", "STAPLES": "XDWS.DE", "BITCOIN": "IB1T.DE", "CRYPTO EQUITY": "DAVV.DE", "STOXX SHORT": "LSK8.DE",
 "COPPER LVG": "3HCL.MI", "PALLADIUM LVG": "2PAL.MI", "NASDAQ 3X": "QQQ3.MI", "S&P 500 SHORT": "DXS3.DE",
 "EURO STOXX 2X": "LYMZ.DE"}
SKIP_IN_STRATEGY = {"OVERNIGHT CASH"}          # money market: nothing to capture
print(len(TICKERS), "ETFs")

## 1. Download: daily closes (official) + hourly bars (last ~2 years)

In [ ]:
def download(interval, **kw):
    for attempt in range(3):
        try:
            d = yf.download(list(TICKERS.values()), interval=interval, auto_adjust=False, group_by="ticker",
                            progress=False, threads=True, **kw)
            if d is not None and not d.empty:
                return d
        except Exception as e:
            print("retry:", type(e).__name__, e)
        time.sleep(10)
    raise RuntimeError("Yahoo returned no data, try again in a few minutes")

hourly = download("60m", period="730d")
daily = download("1d", period="800d")
print("hourly bars:", hourly.index.min(), "->", hourly.index.max())
print("daily closes:", daily.index.min().date(), "->", daily.index.max().date())

## 2. One row per ETF and day
- `prev_close`: official close of the previous trading day = **our execution price**
- `p13`: price at 13:00 on day T (what we can see when we decide)
- `morning_%`: prev_close → 13:00 (known when we decide) · `day_%`: prev_close → close of T (what we actually earn on day T)
- `afternoon_%`: 13:00 → close (does the morning move continue or reverse?)
- `t2_%`, `t5_%`: prev_close → close 2 and 5 sessions later (our valuations are 2 sessions after each trading day)

In [ ]:
today = pd.Timestamp.now(tz=TZ).normalize().tz_localize(None)
rows = []
for code, sym in TICKERS.items():
    try:
        h = hourly[sym][["Close", "Volume"]].dropna(subset=["Close"])
        d = daily[sym]["Close"].dropna()
    except KeyError:
        print("no data:", code, sym); continue
    if h.empty or d.empty:
        print("no data:", code, sym); continue
    h = h.tz_convert(TZ) if h.index.tz is not None else h.tz_localize("UTC").tz_convert(TZ)
    h = h[h.index.hour < DECISION_HOUR]                       # bars that end by 13:00
    day = h.index.tz_localize(None).normalize()
    p13 = h["Close"].groupby(day).last()
    vol_morning = h["Volume"].groupby(day).sum()
    d.index = pd.DatetimeIndex(d.index).tz_localize(None).normalize()
    d = d[d.index < today]                                    # only finished days
    df = pd.DataFrame({"close": d, "prev_close": d.shift(1), "close_t2": d.shift(-2), "close_t5": d.shift(-5)})
    df = df.join(p13.rename("p13"), how="inner").join(vol_morning.rename("morning_volume"), how="left")
    df = df.dropna(subset=["prev_close", "p13", "close"])
    df["code"] = code
    rows.append(df.reset_index(names="date"))

T = pd.concat(rows, ignore_index=True)
T["morning_%"] = (T["p13"] / T["prev_close"] - 1) * 100
T["day_%"] = (T["close"] / T["prev_close"] - 1) * 100
T["afternoon_%"] = (T["close"] / T["p13"] - 1) * 100
T["t2_%"] = (T["close_t2"] / T["prev_close"] - 1) * 100
T["t5_%"] = (T["close_t5"] / T["prev_close"] - 1) * 100
T = T[T["morning_%"].abs() < 25]                              # drop obvious data errors (splits etc.)
T.to_csv("intraday_morning_test.csv", index=False)
print(T["code"].nunique(), "ETFs,", T["date"].nunique(), "days,", len(T), "rows")
T.head()

## 3. Does the morning move survive until the close?

In [ ]:
def kept(g):
    slope = np.polyfit(g["morning_%"], g["day_%"], 1)[0] if len(g) > 30 else np.nan
    return pd.Series({
        "days": len(g),
        "corr_morning_day": round(g["morning_%"].corr(g["day_%"]), 2),
        "share_of_morning_kept": round(slope, 2),          # 1.0 = the whole morning move is kept at the close
        "same_sign_at_close_%": round((np.sign(g["morning_%"]) == np.sign(g["day_%"])).mean() * 100, 1),
        "corr_morning_afternoon": round(g["morning_%"].corr(g["afternoon_%"]), 2),   # < 0 = afternoon reversal
    })

valid = T[~T["code"].isin(SKIP_IN_STRATEGY)]
print("All ETFs together:"); print(kept(valid).to_string())
per_etf = valid.groupby("code").apply(kept).sort_values("share_of_morning_kept", ascending=False)
per_etf

## 4. Strategy test: buy the morning winners at yesterday's close
Every day: rank the ETFs by `morning_%` and buy the top 5 (equal weight) at the T-1 close.
Compare with the average of all ETFs (= no information) and with the bottom 5.
Gains measured at the close of T, and 2 and 5 sessions later.

In [ ]:
def daily_spread(g, n=5):
    g = g.dropna(subset=["morning_%"]).sort_values("morning_%")
    if len(g) < 3 * n:
        return None
    top, bottom = g.tail(n), g.head(n)
    out = {}
    for h in ["day_%", "t2_%", "t5_%"]:
        out[f"top5_{h}"] = top[h].mean()
        out[f"all_{h}"] = g[h].mean()
        out[f"bottom5_{h}"] = bottom[h].mean()
        out[f"excess_top5_{h}"] = top[h].mean() - g[h].mean()
    out["top5_morning_%"] = top["morning_%"].mean()
    return pd.Series(out)

S = valid.groupby("date").apply(daily_spread).dropna(how="all")
res = pd.DataFrame({h: {"avg top5 %": S[f"top5_{h}"].mean(), "avg all ETFs %": S[f"all_{h}"].mean(),
                        "avg bottom5 %": S[f"bottom5_{h}"].mean(), "avg excess top5 %": S[f"excess_top5_{h}"].mean(),
                        "top5 beats average (% of days)": (S[f"excess_top5_{h}"] > 0).mean() * 100}
                    for h in ["day_%", "t2_%", "t5_%"]}).round(2)
print(len(S), "days. Average morning move of the top 5 at 13:00:", round(S["top5_morning_%"].mean(), 2), "%")
res

## 5. Absolute version: buy an ETF only if it is clearly up in the morning
For different thresholds: when an ETF is up more than X% at 13:00, what do we earn by buying it at the T-1 close?

In [ ]:
rows = []
for x in [0.5, 1.0, 1.5, 2.0, 3.0]:
    up = valid[valid["morning_%"] > x]
    dn = valid[valid["morning_%"] < -x]
    rows.append({"threshold_%": x, "cases_up": len(up), "avg_day_%_when_up": up["day_%"].mean(),
                 "still_up_at_close_%": (up["day_%"] > 0).mean() * 100, "avg_t2_%_when_up": up["t2_%"].mean(),
                 "cases_down": len(dn), "avg_day_%_when_down": dn["day_%"].mean(),
                 "still_down_at_close_%": (dn["day_%"] < 0).mean() * 100})
pd.DataFrame(rows).round(2)

## 6. Save and send the results

In [ ]:
per_etf.to_csv("intraday_per_etf.csv")
S.to_csv("intraday_daily_strategy.csv")
try:
    from google.colab import files
    for f in ["intraday_morning_test.csv", "intraday_per_etf.csv", "intraday_daily_strategy.csv"]:
        files.download(f)
    print("Files downloaded: send intraday_morning_test.csv back for the analysis.")
except ImportError:
    print("Not on Colab: the CSV files are in the current folder.")

## Notes for reading the results
- If `share_of_morning_kept` is close to **1** and `same_sign_at_close_%` is high, the morning move mostly stays until the close: buying at yesterday's close the ETFs that are already up captures most of that move.
- If the top 5 also beat the average at **t2** and **t5**, the gain is not reversed before our valuation days.
- This uses a feature of the competition (execution at the previous close), not something possible in a real market. If we use it, we say so openly in the report.
- Hourly data from Yahoo covers only ~2 years and some ETFs trade rarely (their 13:00 price can be stale): check `morning_volume`.